# ARC VLM grid inference

Loads the Qwen3-VL adapter produced by `arc-vlm-grid-sft`, runs two direct output-grid generations per ARC-AGI-2 test query across four L4 GPUs, validates the official schema, and writes `/kaggle/working/submission.json`.

The notebook is fully offline.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

wheelhouse = Path("/kaggle/input/datasets/aishikai/offline-unsloth-trl-wheelhouse-py312-cu128")
assert (wheelhouse / "requirements.in").exists(), "Offline wheelhouse is not attached"
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-index",
    "--find-links", str(wheelhouse), "-r", str(wheelhouse / "requirements.in"),
], check=True)
print("Offline dependencies installed.")


In [ ]:
from pathlib import Path
Path('arc_vlm_data.py').write_text('"""Build rendered, exact-grid ARC vision SFT episodes."""\n\nimport json\nfrom pathlib import Path\n\nfrom datasets import Dataset, Image, Sequence\nfrom PIL import Image as PILImage, ImageDraw, ImageFont\n\n\nPALETTE = [\n    (0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64), (255, 220, 0),\n    (170, 170, 170), (240, 18, 190), (255, 133, 27), (127, 219, 255), (135, 12, 37),\n]\n\n\ndef validate_grid(grid):\n    if not isinstance(grid, list) or not grid or not isinstance(grid[0], list) or not grid[0]:\n        raise ValueError("grid must be a non-empty list of rows")\n    width = len(grid[0])\n    if len(grid) > 30 or width > 30 or any(len(row) != width for row in grid):\n        raise ValueError("grid must be rectangular and at most 30x30")\n    if any(not isinstance(value, int) or not 0 <= value <= 9 for row in grid for value in row):\n        raise ValueError("grid colors must be integers 0..9")\n    return grid\n\n\ndef grid_text(grid):\n    validate_grid(grid)\n    return "\\n".join("".join(str(value) for value in row) for row in grid)\n\n\ndef task_prompt(demos, query):\n    parts = [\n        "Infer the exact output for QUERY. Use the image for whole-scene spatial structure and the exact grids below for coordinates and colors.",\n    ]\n    for index, example in enumerate(demos, 1):\n        inp, out = validate_grid(example["input"]), validate_grid(example["output"])\n        parts.append(f"DEMO {index} INPUT ({len(inp)}x{len(inp[0])}):\\n{grid_text(inp)}")\n        parts.append(f"DEMO {index} OUTPUT ({len(out)}x{len(out[0])}):\\n{grid_text(out)}")\n    query = validate_grid(query)\n    parts.append(f"QUERY INPUT ({len(query)}x{len(query[0])}):\\n{grid_text(query)}")\n    parts.append(\'Return only JSON in the form {"output":[[...]]}.\')\n    return "\\n\\n".join(parts)\n\n\ndef _grid_image(grid, size=210):\n    grid = validate_grid(grid)\n    height, width = len(grid), len(grid[0])\n    cell = max(3, min(size // height, size // width))\n    image = PILImage.new("RGB", (width * cell + 1, height * cell + 1), (64, 64, 64))\n    draw = ImageDraw.Draw(image)\n    for row, values in enumerate(grid):\n        for col, value in enumerate(values):\n            x, y = col * cell, row * cell\n            draw.rectangle((x, y, x + cell - 1, y + cell - 1), fill=PALETTE[value])\n    return image\n\n\ndef _pair_panel(label, inp, out=None):\n    font = ImageFont.load_default()\n    panel = PILImage.new("RGB", (500, 255), "white")\n    draw = ImageDraw.Draw(panel)\n    draw.text((8, 8), label, fill="black", font=font)\n    draw.text((8, 28), "INPUT", fill="black", font=font)\n    left = _grid_image(inp)\n    panel.paste(left, (8, 45))\n    if out is not None:\n        draw.text((258, 28), "OUTPUT", fill="black", font=font)\n        right = _grid_image(out)\n        panel.paste(right, (258, 45))\n    else:\n        draw.text((258, 100), "?", fill="black", font=font)\n    return panel\n\n\ndef render_task(demos, query):\n    panels = [_pair_panel(f"DEMO {index}", row["input"], row["output"]) for index, row in enumerate(demos, 1)]\n    panels.append(_pair_panel("QUERY", query))\n    rows = (len(panels) + 1) // 2\n    image = PILImage.new("RGB", (1010, rows * 265), (224, 224, 224))\n    for index, panel in enumerate(panels):\n        image.paste(panel, ((index % 2) * 510, (index // 2) * 265))\n    if max(image.size) > 1280:\n        scale = 1280 / max(image.size)\n        image = image.resize((round(image.width * scale), round(image.height * scale)), PILImage.Resampling.NEAREST)\n    return image\n\n\ndef episode(task_id, episode_id, demos, query, output, image_dir):\n    output = validate_grid(output)\n    path = image_dir / f"{task_id}_{episode_id}.png"\n    render_task(demos, query).save(path, optimize=True)\n    return {\n        "task_id": task_id,\n        "episode_id": episode_id,\n        "images": [str(path)],\n        "prompt": [{\n            "role": "user",\n            "content": [\n                {"type": "image", "image": str(path)},\n                {"type": "text", "text": task_prompt(demos, query)},\n            ],\n        }],\n        "completion": [{\n            "role": "assistant",\n            "content": [{"type": "text", "text": json.dumps({"output": output}, separators=(",", ":"))}],\n        }],\n        "output": output,\n    }\n\n\ndef build_split(challenges, solutions, image_dir, leave_one_out):\n    image_dir.mkdir(parents=True, exist_ok=True)\n    rows = []\n    for task_id, task in challenges.items():\n        train = task["train"]\n        if leave_one_out:\n            for index, held_out in enumerate(train):\n                demos = train[:index] + train[index + 1:]\n                rows.append(episode(task_id, f"demo_{index}", demos, held_out["input"], held_out["output"], image_dir))\n        for index, test in enumerate(task["test"]):\n            rows.append(episode(task_id, f"test_{index}", train, test["input"], solutions[task_id][index], image_dir))\n    return rows\n\n\ndef build_datasets(competition_dir, output_dir):\n    competition_dir, output_dir = Path(competition_dir), Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    load = lambda name: json.loads((competition_dir / name).read_text())\n    train_rows = build_split(\n        load("arc-agi_training_challenges.json"), load("arc-agi_training_solutions.json"),\n        output_dir / "images/train", leave_one_out=True,\n    )\n    eval_rows = build_split(\n        load("arc-agi_evaluation_challenges.json"), load("arc-agi_evaluation_solutions.json"),\n        output_dir / "images/eval", leave_one_out=False,\n    )\n    for name, rows in (("train", train_rows), ("eval", eval_rows)):\n        dataset = Dataset.from_list(rows).cast_column("images", Sequence(Image()))\n        dataset.save_to_disk(output_dir / name)\n    return len(train_rows), len(eval_rows)\n')
print('Wrote arc_vlm_data.py')


In [ ]:
from pathlib import Path
Path('evaluate.py').write_text('"""Run exact-grid validation for the trained vision adapter."""\n\nimport json\nimport os\nfrom pathlib import Path\n\nos.environ.setdefault("HF_HUB_OFFLINE", "1")\nos.environ.setdefault("TRANSFORMERS_OFFLINE", "1")\n\n\ndef parse_output(text):\n    start = text.find("{")\n    if start < 0:\n        raise ValueError("no JSON object")\n    value, _ = json.JSONDecoder().raw_decode(text[start:])\n    grid = value.get("output") if isinstance(value, dict) else None\n    if not isinstance(grid, list) or not grid or any(not isinstance(row, list) for row in grid):\n        raise ValueError("missing output grid")\n    width = len(grid[0])\n    if not 1 <= len(grid) <= 30 or not 1 <= width <= 30:\n        raise ValueError("invalid grid size")\n    if any(len(row) != width for row in grid):\n        raise ValueError("non-rectangular grid")\n    if any(type(value) is not int or not 0 <= value <= 9 for row in grid for value in row):\n        raise ValueError("invalid color")\n    return grid\n\n\ndef clean_messages(messages):\n    return [{\n        "role": message["role"],\n        "content": [{key: value for key, value in item.items() if value is not None}\n                    for item in message["content"]],\n    } for message in messages]\n\n\ndef main():\n    import torch\n    from datasets import load_from_disk\n    from unsloth import FastVisionModel\n\n    model, processor = FastVisionModel.from_pretrained(\n        model_name="outputs/adapters",\n        max_seq_length=8192,\n        dtype=None,\n        load_in_4bit=True,\n        local_files_only=True,\n    )\n    FastVisionModel.for_inference(model)\n\n    @torch.inference_mode()\n    def predict(row):\n        messages = clean_messages(row["prompt"])\n        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        inputs = processor(text=[text], images=row["images"], return_tensors="pt").to(model.device)\n        generated = model.generate(\n            **inputs,\n            max_new_tokens=128 if os.environ.get("ARC_VLM_SMOKE_TEST") == "1" else 2304,\n            do_sample=False,\n            use_cache=True,\n            pad_token_id=processor.tokenizer.eos_token_id,\n        )\n        return processor.tokenizer.decode(\n            generated[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True\n        )\n\n    dataset = load_from_disk("data/eval")\n    if os.environ.get("ARC_VLM_SMOKE_TEST") == "1":\n        dataset = dataset.select(range(2))\n    records = []\n    for index, row in enumerate(dataset):\n        text = predict(row)\n        try:\n            prediction = parse_output(text)\n            error = None\n        except (json.JSONDecodeError, TypeError, ValueError) as exc:\n            prediction, error = None, str(exc)\n        records.append({\n            "task_id": row["task_id"],\n            "episode_id": row["episode_id"],\n            "correct": prediction == row["output"],\n            "prediction": prediction,\n            "expected": row["output"],\n            "raw": text,\n            "error": error,\n        })\n        if (index + 1) % 10 == 0:\n            print(f"{index + 1}/{len(dataset)}")\n\n    Path("outputs").mkdir(exist_ok=True)\n    Path("outputs/eval_records.json").write_text(json.dumps(records))\n    correct = sum(record["correct"] for record in records)\n    print({"exact": correct, "total": len(records), "accuracy": correct / len(records)})\n\n\nif __name__ == "__main__":\n    main()\n')
print('Wrote evaluate.py')


In [ ]:
from pathlib import Path
Path('infer_ddp.py').write_text('"""Generate two ARC output-grid attempts on one Kaggle GPU shard."""\n\nimport json\nimport os\nimport random\nfrom pathlib import Path\n\nos.environ.setdefault("HF_HUB_OFFLINE", "1")\nos.environ.setdefault("TRANSFORMERS_OFFLINE", "1")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nimport numpy as np\nimport torch\n\n\nRANK = int(os.environ["LOCAL_RANK"])\nWORLD_SIZE = int(os.environ["WORLD_SIZE"])\nSEED = 3407\nMAX_SEQ_LENGTH = 8192\nMAX_NEW_TOKENS = 2048\nif WORLD_SIZE != 4:\n    raise RuntimeError(f"Expected four GPU processes, got {WORLD_SIZE}")\ntorch.cuda.set_device(RANK)\n\nfrom unsloth import FastVisionModel\n\nfrom arc_vlm_data import render_task, task_prompt, validate_grid\nfrom evaluate import parse_output\n\n\nconfig = json.loads(Path("inference_paths.json").read_text())\nchallenges = json.loads(Path(config["test_file"]).read_text())\nmodel, processor = FastVisionModel.from_pretrained(\n    model_name=config["adapter"],\n    max_seq_length=MAX_SEQ_LENGTH,\n    dtype=None,\n    load_in_4bit=True,\n    local_files_only=True,\n)\nFastVisionModel.for_inference(model)\nprocessor.tokenizer.eos_token = "<|im_end|>"\n\n\n@torch.inference_mode()\ndef generate(demos, query, sample, seed):\n    image = render_task(demos, query)\n    messages = [{\n        "role": "user",\n        "content": [\n            {"type": "image"},\n            {"type": "text", "text": task_prompt(demos, query)},\n        ],\n    }]\n    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)\n    torch.manual_seed(seed)\n    kwargs = {\n        "max_new_tokens": MAX_NEW_TOKENS,\n        "do_sample": sample,\n        "use_cache": True,\n        "pad_token_id": processor.tokenizer.eos_token_id,\n    }\n    if sample:\n        kwargs.update(temperature=0.35, top_p=0.95)\n    tokens = model.generate(**inputs, **kwargs)\n    raw = processor.tokenizer.decode(\n        tokens[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True\n    )\n    return parse_output(raw), raw\n\n\nrecords = {}\ntask_items = list(challenges.items())[RANK::WORLD_SIZE]\nfor task_number, (task_id, task) in enumerate(task_items, 1):\n    task_records = []\n    for test_index, test in enumerate(task["test"]):\n        query = validate_grid(test["input"])\n        attempts, raw, errors = [], [], []\n        for sample in (False, True):\n            try:\n                grid, text = generate(\n                    task["train"], query, sample,\n                    SEED + int(task_id, 16) + test_index * 2 + int(sample),\n                )\n                attempts.append(grid)\n                raw.append(text)\n                errors.append(None)\n            except (json.JSONDecodeError, RuntimeError, TypeError, ValueError) as exc:\n                attempts.append(query)\n                raw.append("")\n                errors.append(str(exc))\n        task_records.append({\n            "attempt_1": attempts[0],\n            "attempt_2": attempts[1],\n            "raw": raw,\n            "errors": errors,\n        })\n    records[task_id] = task_records\n    print(f"rank={RANK} tasks={task_number}/{len(task_items)}", flush=True)\n\nPath("outputs").mkdir(exist_ok=True)\nPath(f"outputs/shard_{RANK}.json").write_text(json.dumps(records))\n')
print('Wrote infer_ddp.py')


In [ ]:
import json
from pathlib import Path


def find_file(name, preferred=()):
    matches = list(Path("/kaggle/input").glob(f"**/{name}"))
    matches.sort(key=lambda path: (-sum(part in str(path) for part in preferred), len(path.parts)))
    return matches[0] if matches else None


adapter_file = find_file(
    "adapter_model.safetensors", preferred=("arc-vlm-grid-sft", "outputs/adapters")
)
test_file = find_file("arc-agi_test_challenges.json", preferred=("arc-prize-2026",))
sample_file = find_file("sample_submission.json", preferred=("arc-prize-2026",))
assert adapter_file and (adapter_file.parent / "adapter_config.json").exists(), "Training adapter is not attached"
assert adapter_file.stat().st_size > 100_000_000, "Adapter file is unexpectedly small"
assert test_file and sample_file, "ARC Prize 2026 competition files are not attached"

paths = {
    "adapter": str(adapter_file.parent),
    "test_file": str(test_file),
    "sample_file": str(sample_file),
}
Path("inference_paths.json").write_text(json.dumps(paths))
print(paths, {"adapter_bytes": adapter_file.stat().st_size})


In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "torch.distributed.run", "--standalone",
    "--nproc_per_node=4", "infer_ddp.py",
], check=True)
print("All inference shards completed.")


In [ ]:
import json
from pathlib import Path

paths = json.loads(Path("inference_paths.json").read_text())
sample = json.loads(Path(paths["sample_file"]).read_text())
records = {}
for rank in range(4):
    records.update(json.loads(Path(f"outputs/shard_{rank}.json").read_text()))

assert set(records) == set(sample)
submission = {}
diagnostics = {}
for task_id, expected_rows in sample.items():
    assert len(records[task_id]) == len(expected_rows)
    submission[task_id] = []
    diagnostics[task_id] = []
    for row in records[task_id]:
        prediction = {"attempt_1": row["attempt_1"], "attempt_2": row["attempt_2"]}
        assert set(prediction) == {"attempt_1", "attempt_2"}
        for grid in prediction.values():
            assert isinstance(grid, list) and grid and isinstance(grid[0], list) and grid[0]
            width = len(grid[0])
            assert len(grid) <= 30 and width <= 30 and all(len(line) == width for line in grid)
            assert all(type(value) is int and 0 <= value <= 9 for line in grid for value in line)
        submission[task_id].append(prediction)
        diagnostics[task_id].append({"raw": row["raw"], "errors": row["errors"]})

submission_file = Path("/kaggle/working/submission.json")
submission_file.write_text(json.dumps(submission, separators=(",", ":")))
Path("/kaggle/working/inference_diagnostics.json").write_text(json.dumps(diagnostics))
print({
    "tasks": len(submission),
    "outputs": sum(len(rows) for rows in submission.values()),
    "bytes": submission_file.stat().st_size,
    "file": str(submission_file),
})
